# 01. Tối ưu không ràng buộc — Genetic Algorithm tìm Global Optimum

In [ ]:
import time

import numpy as np
import sympy
from IPython.display import Markdown, display

print(f"numpy {np.__version__} | sympy {sympy.__version__}")

In [ ]:
# ============================================================
# GENETIC ALGORITHM CHO TỐI ƯU KHÔNG RÀNG BUỘC
# ============================================================
#
# Khác với GA có ràng buộc (notebook 02): ở đây fitness = f(x) trực
# tiếp, không cần quy tắc khả thi Deb hay ngưỡng epsilon giảm dần.
#
# Miền tìm kiếm [lower, upper] đóng vai trò MIỀN XÁC ĐỊNH của bài toán
# tối ưu toàn cục, nên cá thể được CẮT (clip) về đúng miền này sau mỗi
# phép lai ghép / đột biến — khác GA có ràng buộc, nơi hộp chỉ dùng để
# khởi tạo quần thể chứ không giới hạn cá thể.


def genetic_algorithm_unconstrained(
    objective,
    bounds,

    population_size=500,
    generations=100,

    crossover_rate=0.9,
    mutation_rate=0.15,
    mutation_scale=0.08,

    elite_size=2,
    tournament_size=3,

    seed=42,
):
    """GA mã hóa số thực cho bài toán tối ưu KHÔNG ràng buộc, tìm cực tiểu
    toàn cục của f(x) trong hộp [lower, upper]."""

    rng = np.random.default_rng(seed)

    bounds = np.asarray(bounds, dtype=float)

    lower = bounds[:, 0]
    upper = bounds[:, 1]

    variable_range = upper - lower
    n_variables = len(bounds)

    def clip(pop):
        return np.clip(pop, lower, upper)

    # --------------------------------------------------------
    # Initial population
    # --------------------------------------------------------

    population = clip(
        rng.uniform(lower, upper, size=(population_size, n_variables))
    )

    def evaluate(pop):
        return np.array([objective(individual) for individual in pop])

    # --------------------------------------------------------
    # Tournament selection (theo thứ hạng fitness)
    # --------------------------------------------------------

    def tournament_selection(rank):

        indices = rng.integers(0, population_size, size=tournament_size)

        best_index = indices[np.argmin(rank[indices])]

        return population[best_index].copy()

    # --------------------------------------------------------
    # Blend crossover
    # --------------------------------------------------------

    def crossover(parent1, parent2):

        if rng.random() > crossover_rate:
            return parent1.copy(), parent2.copy()

        alpha = rng.uniform(-0.25, 1.25, size=n_variables)

        child1 = alpha * parent1 + (1 - alpha) * parent2
        child2 = alpha * parent2 + (1 - alpha) * parent1

        return child1, child2

    # --------------------------------------------------------
    # Gaussian mutation
    # --------------------------------------------------------

    def mutate(child):

        mutation_mask = rng.random(n_variables) < mutation_rate

        if np.any(mutation_mask):
            child[mutation_mask] += rng.normal(
                loc=0,
                scale=mutation_scale * variable_range[mutation_mask],
            )

        return child

    # --------------------------------------------------------
    # Evolution
    # --------------------------------------------------------

    history = []

    start_time = time.perf_counter()

    fitness = evaluate(population)

    best_index = int(np.argmin(fitness))
    best_solution = population[best_index].copy()
    best_fitness = fitness[best_index]

    for generation in range(generations):

        order = np.argsort(fitness)

        rank = np.empty(population_size, dtype=np.int64)
        rank[order] = np.arange(population_size)

        current_best = int(np.argmin(fitness))
        if fitness[current_best] < best_fitness:
            best_fitness = fitness[current_best]
            best_solution = population[current_best].copy()

        history.append(best_fitness)

        # Elitism
        new_population = [
            population[i].copy()
            for i in order[:elite_size]
        ]

        # Sinh thế hệ tiếp theo
        while len(new_population) < population_size:

            parent1 = tournament_selection(rank)
            parent2 = tournament_selection(rank)

            child1, child2 = crossover(parent1, parent2)

            new_population.append(mutate(child1))

            if len(new_population) < population_size:
                new_population.append(mutate(child2))

        population = clip(np.asarray(new_population))
        fitness = evaluate(population)

    current_best = int(np.argmin(fitness))
    if fitness[current_best] < best_fitness:
        best_fitness = fitness[current_best]
        best_solution = population[current_best].copy()

    elapsed_time = time.perf_counter() - start_time

    # Thế hệ sớm nhất mà nghiệm tốt nhất (best_fitness cuối cùng) đã đạt
    # được — chỉ là một chỉ số báo cáo, KHÔNG dừng vòng lặp sớm.
    generations_run = int(np.argmin(history)) + 1

    return {
        "x": best_solution,
        "fun": best_fitness,
        "time": elapsed_time,
        "history": history,
        "generations": generations,
        "generations_run": generations_run,
        "seed": seed,
    }

In [ ]:
def make_objective(expr, variables):
    """Chuyển biểu thức SymPy thành hàm mục tiêu NumPy cho GA.

    Giá trị không hữu hạn (NaN, tràn số...) được quy về +inf để cá thể
    tương ứng bị loại tự nhiên trong quá trình chọn lọc."""

    objective_raw = sympy.lambdify(variables, expr, modules="numpy")

    def objective(v):
        try:
            value = float(np.asarray(objective_raw(*v)).reshape(()))
            if np.isfinite(value):
                return value
        except Exception:
            pass
        return np.inf

    return objective

In [ ]:
# ------------------------------------------------------------------
# Hiển thị dạng ký hiệu toán học (LaTeX)
# ------------------------------------------------------------------

def _num(value, digits=10):
    """Số dạng LaTeX; chuyển sang ký hiệu khoa học khi quá lớn hoặc quá nhỏ."""
    if not np.isfinite(value):
        return r"\infty" if value > 0 else r"-\infty"
    if value != 0 and (abs(value) >= 1e6 or abs(value) < 1e-4):
        mantissa, exponent = f"{value:.4e}".split("e")
        return mantissa + r" \times 10^{" + str(int(exponent)) + "}"
    return f"{value:.{digits}f}"

In [ ]:
def _num_bound(value):
    """Số dạng LaTeX cho cận tìm kiếm — bỏ số 0 thừa: -10 thay vì -10.0000."""
    text = _num(value, 4)
    if "." in text and "times" not in text:
        text = text.rstrip("0").rstrip(".")
    return text

In [ ]:
def show_problem(objective_expr, variables, bounds):
    """Phát biểu bài toán tối ưu không ràng buộc và miền tìm kiếm."""
    bien = ", ".join(sympy.latex(v) for v in variables)
    mien = ", \\ ".join(
        f"{_num_bound(b[0])} \\le {sympy.latex(v)} \\le {_num_bound(b[1])}"
        for v, b in zip(variables, bounds)
    )
    display(Markdown(
        "$$\n\\underset{" + bien + r"}{\text{minimize}} \quad f\left("
        + bien + r"\right) = " + sympy.latex(objective_expr) + "\n$$\n\n"
        + "Miền tìm kiếm: $" + mien + "$"
    ))

In [ ]:
def show_result(result, variables):
    """Kết quả GA gói trên MỘT hàng, ngay dưới dòng miền tìm kiếm: thời
    gian hội tụ, số thế hệ thỏa mãn, giá trị tối ưu và tọa độ nghiệm."""

    bien = "(" + ", ".join(sympy.latex(v) for v in variables) + ")"
    toado = "(" + ", \\ ".join(_num(x) for x in result["x"]) + ")"

    display(Markdown(
        "| Thời gian hội tụ (s) | Số thế hệ thỏa mãn | $f^{*}$ | $" + bien + "$ |\n"
        "|---|---|---|---|\n"
        "| $" + _num(result["time"], 6) + "$ | $" + str(result["generations_run"])
        + "$ | $" + _num(result["fun"]) + "$ | $" + toado + "$ |"
    ))

In [ ]:
x, y = sympy.symbols("x y")
variables_xy = [x, y]

# Tham số GA dùng cho toàn bộ benchmark bên dưới.
POPULATION_SIZE = 500
GENERATIONS = 100
SEED = 42

# Năm hàm benchmark kinh điển, đặt thủ công (không qua parser) —
# miền tìm kiếm lấy theo giá trị chuẩn thường dùng trong tài liệu.
BENCHMARKS = {
    "Easom": {
        "expr": (
            -sympy.cos(x) * sympy.cos(y)
            * sympy.exp(-((x - sympy.pi) ** 2 + (y - sympy.pi) ** 2))
        ),
        "bounds": [(-100.0, 100.0), (-100.0, 100.0)],
        "optimum": r"f(\pi, \pi) = -1",
    },
    "Rastrigin": {
        "expr": (
            20 + x**2 - 10 * sympy.cos(2 * sympy.pi * x)
            + y**2 - 10 * sympy.cos(2 * sympy.pi * y)
        ),
        "bounds": [(-5.12, 5.12), (-5.12, 5.12)],
        "optimum": "f(0, 0) = 0",
    },
    "Rosenbrock": {
        "expr": (1 - x) ** 2 + 100 * (y - x**2) ** 2,
        "bounds": [(-5.0, 10.0), (-5.0, 10.0)],
        "optimum": "f(1, 1) = 0",
    },
    "Ackley": {
        "expr": (
            -20 * sympy.exp(-0.2 * sympy.sqrt(0.5 * (x**2 + y**2)))
            - sympy.exp(0.5 * (sympy.cos(2 * sympy.pi * x) + sympy.cos(2 * sympy.pi * y)))
            + 20 + sympy.E
        ),
        "bounds": [(-32.768, 32.768), (-32.768, 32.768)],
        "optimum": "f(0, 0) = 0",
    },
    "Schwefel": {
        "expr": (
            418.9829 * 2
            - x * sympy.sin(sympy.sqrt(sympy.Abs(x)))
            - y * sympy.sin(sympy.sqrt(sympy.Abs(y)))
        ),
        "bounds": [(-500.0, 500.0), (-500.0, 500.0)],
        "optimum": r"f(420.9687, 420.9687) \approx 0",
    },
}

In [ ]:
def run_benchmark(name):
    """Chạy GA MỘT lần trên một hàm benchmark: phát biểu bài toán rồi in
    hàng kết quả."""

    spec = BENCHMARKS[name]

    expr = spec["expr"]
    bounds = spec["bounds"]

    objective = make_objective(expr, variables_xy)

    display(Markdown(f"### {name} — nghiệm đúng: ${spec['optimum']}$"))
    show_problem(expr, variables_xy, bounds)

    result = genetic_algorithm_unconstrained(
        objective, bounds,
        population_size=POPULATION_SIZE,
        generations=GENERATIONS,
        seed=SEED,
    )

    show_result(result, variables_xy)

    return result

## Easom

In [ ]:
result_easom = run_benchmark("Easom")

## Rastrigin

In [ ]:
result_rastrigin = run_benchmark("Rastrigin")

## Rosenbrock

In [ ]:
result_rosenbrock = run_benchmark("Rosenbrock")

## Ackley

In [ ]:
result_ackley = run_benchmark("Ackley")

## Schwefel

In [ ]:
result_schwefel = run_benchmark("Schwefel")